# Neural Networks with PyTorch\n\n**A Practical Introduction to Deep Learning**

## Learning Objectives\n\nBy the end of this lecture, you will understand:\n\n- **What neural networks are** and how they extend logistic regression\n- **How neural networks learn** through backpropagation and gradient descent\n- **How to implement** a neural network using PyTorch\n- **How to train and evaluate** a neural network for image classification

## From Logistic Regression to Neural Networks\n\n**Logistic Regression** (simplest neural network):\n- Single layer: input → output\n- Linear decision boundaries\n\n**Neural Networks** add hidden layers:\n- Input → **Hidden Layer(s)** → Output\n- Can learn non-linear patterns\n- **Universal approximator** - can model any function!

## Importing Libraries

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\n# PyTorch imports\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader, TensorDataset\n\ntorch.manual_seed(42)\nnp.random.seed(42)

# The MNIST Dataset

## What is MNIST?\n\n- **60,000 training** + **10,000 test** images\n- Handwritten digits (0-9)\n- 28×28 pixel grayscale images\n- The 'Hello World' of deep learning

In [ ]:
# Load from provided CSV files (no internet needed)\ntrain_df = pd.read_csv('mnist_train.csv')\ny_train = train_df.iloc[:, 0].values\nx_train = train_df.iloc[:, 1:].values\n\ntest_df = pd.read_csv('mnist_test.csv')\ny_test = test_df.iloc[:, 0].values\nx_test = test_df.iloc[:, 1:].values\n\nprint(f'Train: {x_train.shape}, Test: {x_test.shape}')

## Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))\nfor i, ax in enumerate(axes.flat):\n    ax.imshow(x_train[i].reshape(28, 28), cmap='gray')\n    ax.set_title(f'Label: {y_train[i]}')\n    ax.axis('off')\nplt.tight_layout()\nplt.show()

# Building the Neural Network

## Network Architecture\n\n**Input**: 784 neurons (28×28 pixels)\n\n**Hidden Layer 1**: 256 neurons + ReLU + Dropout(0.45)\n\n**Hidden Layer 2**: 256 neurons + ReLU + Dropout(0.45)\n\n**Output**: 10 neurons (digits 0-9) + Softmax

In [ ]:
# Preprocess data\nx_train = x_train.astype('float32') / 255.0\nx_test = x_test.astype('float32') / 255.0\n\n# Convert to tensors\nx_train_tensor = torch.FloatTensor(x_train)\ny_train_tensor = torch.LongTensor(y_train)\nx_test_tensor = torch.FloatTensor(x_test)\ny_test_tensor = torch.LongTensor(y_test)

## Define the PyTorch Model

In [ ]:
class NeuralNetwork(nn.Module):\n    def __init__(self):\n        super(NeuralNetwork, self).__init__()\n        self.fc1 = nn.Linear(784, 256)\n        self.dropout1 = nn.Dropout(0.45)\n        self.fc2 = nn.Linear(256, 256)\n        self.dropout2 = nn.Dropout(0.45)\n        self.fc3 = nn.Linear(256, 10)\n    \n    def forward(self, x):\n        x = torch.relu(self.fc1(x))\n        x = self.dropout1(x)\n        x = torch.relu(self.fc2(x))\n        x = self.dropout2(x)\n        x = self.fc3(x)\n        return x\n\nmodel = NeuralNetwork()\nprint(model)

## Training Setup\n\n**Loss Function**: CrossEntropyLoss (combines Softmax + NLL)\n\n**Optimizer**: Adam (adaptive learning rate)\n\n**Learning Rate**: 0.001\n\n**Batch Size**: 128\n\n**Epochs**: 20

In [ ]:
# Create data loaders\ntrain_dataset = TensorDataset(x_train_tensor, y_train_tensor)\ntrain_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)\n\n# Define loss and optimizer\ncriterion = nn.CrossEntropyLoss()\noptimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the Model

In [ ]:
# Training loop\nepochs = 20\nmodel.train()\n\nfor epoch in range(epochs):\n    running_loss = 0.0\n    correct = 0\n    total = 0\n    \n    for inputs, labels in train_loader:\n        # Zero gradients\n        optimizer.zero_grad()\n        \n        # Forward pass\n        outputs = model(inputs)\n        loss = criterion(outputs, labels)\n        \n        # Backward pass and optimize\n        loss.backward()\n        optimizer.step()\n        \n        # Statistics\n        running_loss += loss.item()\n        _, predicted = torch.max(outputs.data, 1)\n        total += labels.size(0)\n        correct += (predicted == labels).sum().item()\n    \n    accuracy = 100 * correct / total\n    print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {accuracy:.2f}%')

# Model Evaluation

In [ ]:
# Evaluate on test set\nmodel.eval()\ncorrect = 0\ntotal = 0\n\nwith torch.no_grad():\n    outputs = model(x_test_tensor)\n    _, predicted = torch.max(outputs.data, 1)\n    total = y_test_tensor.size(0)\n    correct = (predicted == y_test_tensor).sum().item()\n\naccuracy = 100 * correct / total\nprint(f'Test Accuracy: {accuracy:.2f}%')

## Visualize Predictions

In [ ]:
# Show some predictions\nmodel.eval()\nwith torch.no_grad():\n    sample_outputs = model(x_test_tensor[:10])\n    _, predictions = torch.max(sample_outputs, 1)\n\nfig, axes = plt.subplots(2, 5, figsize=(12, 5))\nfor i, ax in enumerate(axes.flat):\n    ax.imshow(x_test[i].reshape(28, 28), cmap='gray')\n    ax.set_title(f'True: {y_test[i]}, Pred: {predictions[i].item()}')\n    ax.axis('off')\nplt.tight_layout()\nplt.show()

# Summary\n\n**What we learned:**\n- Neural networks extend logistic regression with hidden layers\n- PyTorch provides flexible tools for building neural networks\n- Training involves forward pass, loss calculation, backpropagation, and optimization\n- Dropout and proper initialization help prevent overfitting\n\n**Next steps:**\n- Try different architectures (more layers, different sizes)\n- Experiment with learning rates and batch sizes\n- Implement convolutional neural networks (CNNs) for better image classification

# Acknowledgments\n\nThis notebook was improved by **OpenClaw** as part of an AI-assisted learning project.\n\n**AI Interaction Documentation:**\n- Original TensorFlow/Keras code converted to PyTorch\n- Lecture explanations enhanced for clarity\n- Code structure improved with better comments\n- RISE slide compatibility maintained